# testing swap pricing and sub functions setep by step

In [ ]:
%load_ext autoreload
%autoreload 2
#imports
from dateutil.relativedelta import relativedelta
from rivapy.pricing.interest_rate_swap_pricing import InterestRateSwapPricer
from rivapy.instruments.ir_swap_specification import InterestRateSwapSpecification, IrFixedLegSpecification, IrFloatLegSpecification, IrOISLegSpecification
from rivapy.instruments.components import NotionalStructure, ConstNotionalStructure, VariableNotionalStructure, ResettingNotionalStructure
from rivapy.pricing.pricing_data import (
    InterestRateSwapFloatLegPricingData_rivapy,
    InterestRateSwapLegPricingData_rivapy,
    InterestRateSwapPricingData_rivapy,
)
from rivapy.pricing.pricing_request import InterestRateSwapPricingRequest
from rivapy.pricing.interest_rate_swap_pricing import InterestRateSwapPricer

import datetime as dt
from rivapy.marketdata.curves import DiscountCurve
from rivapy.tools.enums import InterpolationType, ExtrapolationType
import math
from rivapy.tools.datetools import DayCounter

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Discount curve - we use these discount factors to get the present values of both the fixed and floating leg as well as

object_id = "TEST_DC"
refdatedc = dt.datetime(2017, 1, 1)
days_to_maturity = [180, 360, 540]
dates = [refdatedc + dt.timedelta(days=d) for d in days_to_maturity]
# discount factors from constant rate
rates = [0.10, 0.105, 0.11]
df = [math.exp(-r * d / 360) for r, d in zip(rates, days_to_maturity)]
dc = DiscountCurve(
    id=object_id, refdate=refdatedc, dates=dates, df=df, interpolation=InterpolationType.LINEAR, extrapolation=ExtrapolationType.LINEAR
)

In [3]:
#test market data
ref_date = dt.datetime(2017, 1, 1)
dates = [dt.datetime(2017, 1, 1), dt.datetime(2018, 7, 1)]
df = [1.0, 0.9900633419771339]
dc = DiscountCurve(id=object_id, refdate=refdatedc, dates=dates, df=df, interpolation=InterpolationType.LINEAR, extrapolation=ExtrapolationType.LINEAR)

In [4]:
# Create the vectors defining the statdates, enddates, paydates and reset dates
days_to_maturity = [0, 180, 360, 540]
dates = [dt.datetime(2017, 1, 1) + dt.timedelta(d) for d in days_to_maturity]

startdates = dates[:-1]
enddates = dates[1:]
paydates = enddates
resetdates = startdates
refdate = dates[0]

In [5]:
# fixedleg = IrFixedLegSpecification(0.08, notionals, startdates, enddates, paydates, 'EUR', 'Act360')
fixed_leg = IrFixedLegSpecification(
    fixed_rate=0.08,
    obj_id="dummy_fixed_leg",
    notional=100.0,
    start_dates=startdates,
    end_dates=enddates,
    pay_dates=paydates,
    currency="EUR",
    day_count_convention="Act360",
)

spread = 0.00

ns = ConstNotionalStructure(100.0)
# floatleg = IrFloatLegSpecification(notionals, resetdates, startdates, enddates, paydates, 'EUR', 'test_udl',
#                                   'Act360', spread)

float_leg = IrFloatLegSpecification(
    obj_id="dummy_float_leg",
    notional=ns,
    reset_dates=resetdates,
    start_dates=startdates,
    end_dates=enddates,
    rate_start_dates=startdates,
    rate_end_dates=enddates,
    pay_dates=paydates,
    currency="EUR",
    udl_id="test_udl_id",
    fixing_id="test_fixing_id",
    day_count_convention="Act360",
    spread=spread,
)

maturity_date = refdate + dt.timedelta(600)
# ir_swap = InterestRateSwapSpecification('TEST_SWAP', 'DBK', 'COLLATERALIZED', 'EUR', paydates[-1], fixedleg, floatleg)
ir_swap = InterestRateSwapSpecification(
    obj_id="dummy_swap",
    notional=ns,
    issue_date=refdate,
    maturity_date=maturity_date,
    pay_leg=fixed_leg,
    receive_leg=float_leg,
    currency="EUR",
    day_count_convention="EUR",
    issuer="DBK",
    securitization_level="COLLATERALIZED",
)

In [6]:
#Pricing the fixed leg

fixed_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )



In [7]:
#pricing float leg
float_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, float_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace peropd
    )



In [8]:
print(f"float pv: {float_PV}")
print(f"fixed_pv: {fixed_PV}")
print(f"price = {float_PV - fixed_PV}")

float pv: 0.9801955758587282
fixed_pv: 11.758073708880719
price = -10.777878133021991


In [9]:
#Computeing fixed leg annuity, i.e. if fixed ratet = 1
pricing_params = {"set_rate": True, "desired_rate": 1.0}
fixed_leg_annuity = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0, pricing_params  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )

print(f"annuity: {fixed_leg_annuity}")

annuity: 146.975921361009


In [10]:
#compute fair rate swap i.e. float_pv = fixed_pv
# where we want the fixed rate that makes that equation true

fair_swap_rate = float_PV / fixed_leg_annuity
print(f"fair swap rate: {fair_swap_rate}")

fair swap rate: 0.006669089513316449


# OIS swap test

modelled after a plain vanilla IR swap, except floating leg is replaced by an OIS leg due to the different compounding


In [11]:
#ref_date
ref_date = dt.datetime(2017, 1, 1)

#1Y maturity 3M swap, i.e. the floating leg is reset every 3M
#since it is OIS
#1 Year maturity, 3M underlying, daily resetting (Overnight-index-swap)
# start dates of the accrual periods corresponding to the tenor of the underlying index (3 months). The spot lag is set to 0.
start_dates = [ref_date + relativedelta(months=3*i) for i in range(4)]

# reset dates are equal to start dates if spot lag is 0.
reset_dates = start_dates

# the end dates of the accrual periods
end_dates = [x + relativedelta(months=3) for x in start_dates]

# the actual payment dates of the cashflows may differ from the end of the accrual period (e.g. OIS). 
# in the standard case these two sets of dates coincide
#pay_dates = end_dates

print(end_dates)
ns = ConstNotionalStructure(100.0)
spread = 0.00

# # definition of the floating leg
# float_leg =IrFloatLegSpecification(obj_id = 'dummy_float_leg', notional = ns, reset_dates=reset_dates, start_dates=start_dates, end_dates=end_dates,
#                                    rate_start_dates=start_dates, rate_end_dates=end_dates, pay_dates=pay_dates, currency = "EUR", 
#                                    udl_id="test_udl_id", fixing_id="test_fixing_id", day_count_convention="Act365Fixed", spread=spread)

#the difference here is that rate_date arrays are excpected to be 2 dimensional, i.e. keep track of the daily resetting per accrual period


#
daily_rate_start_dates = []   # 2D list: coupon i -> list of daily starts
daily_rate_end_dates   = []   # 2D list: coupon i -> list of daily ends
daily_rate_reset_dates = []   # 2D list: coupon i -> list of reset dates
pay_dates              = []   # 1D list: one pay date per coupon

for i in range(len(start_dates)):

    # 1. Generate the raw daily schedule for this coupon
    # daily_schedule = generate_schedule(start_dates[i], end_dates[i],
    #                                    freq=rateFreq,
    #                                    roll=rateRoll,
    #                                    holidays=rateHolidays,
    #                                    rule="Forward")

    # # 2. Remove duplicates
    # daily_schedule = deduplicate(daily_schedule)

    #for this test we keep it simple and ignore conventions e.g. business day or so. i.e just take every day
    num_days = (end_dates[i] - start_dates[i]).days
    daily_schedule = [start_dates[i] + dt.timedelta(days=j) for j in range(num_days)]

    # 3. Build start/end date pairs for accrual periods
    starts = daily_schedule[:-1]   # all except last
    ends   = daily_schedule[1:]    # all except first

    daily_rate_start_dates.append(starts)
    daily_rate_end_dates.append(ends)

    # 4. Compute reset dates (fixing lag applied to each start)
    # resets = [add_business_days(start, fixingLag, rateHolidays)
    #           for start in starts]
    #assume simple case reset date is the same as start date
    resets = starts
    daily_rate_reset_dates.append(resets)

    # 5. Compute payment date for the coupon
    #pay_date = add_business_days(end_dates[i], payLag, holidays)
    #assume simple case, pay date is end date
    pay_date=end_dates[i]
    pay_dates.append(pay_date)

float_leg = IrOISLegSpecification(obj_id = 'dummy_float_leg', notional = ns, rate_reset_dates=daily_rate_reset_dates, start_dates=start_dates, end_dates=end_dates,
                                   rate_start_dates=daily_rate_start_dates, rate_end_dates=daily_rate_end_dates, pay_dates=pay_dates, currency = "EUR", 
                                   udl_id="test_udl_id", fixing_id="test_fixing_id", day_count_convention="Act365Fixed", rate_day_count_convention="Act365Fixed",spread=spread)

# # definition of the fixed leg
#Note that a fixed rate is given for the specification as it is required. 
#However, for the creation of the bootrstrapped curve, the market quotes are used as the target swap par rate
fixed_leg = IrFixedLegSpecification(fixed_rate = 0.01, obj_id = 'dummy_fixed_leg', notional = 100.0, start_dates=start_dates, 
                                    end_dates=end_dates, pay_dates=pay_dates, currency='EUR', day_count_convention='Act365Fixed')

# # definition of the IR swap
ir_swap = InterestRateSwapSpecification(obj_id="3M_SWAP", notional=ns, issue_date=ref_date, maturity_date=pay_dates[-1],
                                        pay_leg=fixed_leg, receive_leg=float_leg,currency='EUR', day_count_convention="Act365Fixed",
                                        issuer="dummy_issuer", securitization_level="COLLATERALIZED")

[datetime.datetime(2017, 4, 1, 0, 0), datetime.datetime(2017, 7, 1, 0, 0), datetime.datetime(2017, 10, 1, 0, 0), datetime.datetime(2018, 1, 1, 0, 0)]


In [12]:
fixed_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )



In [13]:
#pricing float leg
float_PV = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, float_leg, None, 0  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace peropd
    )



In [14]:
print(f"float pv: {float_PV}")
print(f"fixed_pv: {fixed_PV}")
print(f"price = {float_PV - fixed_PV}")

float pv: 0.6642516297667257
fixed_pv: 0.9958482828862965
price = -0.33159665311957076


In [15]:
#Computeing fixed leg annuity, i.e. if fixed ratet = 1
pricing_params = {"set_rate": True, "desired_rate": 1.0}
fixed_leg_annuity = InterestRateSwapPricer.price_leg(
        refdate, dc, dc, None, fixed_leg, None, 0, pricing_params  # discount  # forward/fixing  # fx_fwd  # leg_spec  # fixing table  # fixing grace period
    )

print(f"annuity: {fixed_leg_annuity}")

annuity: 99.58482828862965


In [16]:
#compute fair rate swap i.e. float_pv = fixed_pv
# where we want the fixed rate that makes that equation true

fair_swap_rate = float_PV / fixed_leg_annuity
print(f"fair swap rate: {fair_swap_rate}")

fair swap rate: 0.006670209119018668


In [17]:
dc.value(refdate, pay_dates[-1])

0.9933573623107214

In [18]:
start_dates

[datetime.datetime(2017, 1, 1, 0, 0),
 datetime.datetime(2017, 4, 1, 0, 0),
 datetime.datetime(2017, 7, 1, 0, 0),
 datetime.datetime(2017, 10, 1, 0, 0)]

In [22]:
rate_dcc = DayCounter(dc.daycounter)
denom = 0
for i in range(len(end_dates)):
    denom +=rate_dcc.yf(start_dates[i], end_dates[i])*dc.value(refdate, end_dates[i])

In [ ]:
#fair swap ois rate
#numerator: PV float leg = In OIS par swaps, the floating leg compounds overnight rates and pays out at termination. Its present value is exactly 1−DF(0,Tn)1−DF(0,Tn) when assuming the notional is 1.
K_ois = (1 - dc.value(refdate, end_dates[-1]))/denom
print(K_ois)


#difference due to unspecific daycount conventions, and roll dates?



0.006670331016714803


In [ ]:
(fair_swap_rate - K_ois)/fair_swap_rate*100

-0.0018274943702645257

: 